# CS412 Add-Drop Recitation

## Machine Learning Concepts & k-Nearest Neighbors

**Author:** Ahmet Emre Küçükatay, Bedirhan Kırtan

**Date:** October 2026

**Email**: ahmet.kucukatay@sabanciuniv.edu,  bedirhan.kirtan@sabanciuniv.edu

In [ ]:
# Make sure you have these libraries
# !pip install matplotlib

# !pip install scikit-learn

# Introduction to Libraries

In this notebook, we will use three commonly used Python libraries for data science and machine learning:  
**NumPy**, **Matplotlib**, and **Scikit-learn**.

---

## NumPy
- Provides efficient numerical operations and array manipulation.
- We will use it to work with feature vectors and calculate distances.

---

## Matplotlib
- A widely used library for data visualization.
- We will use it to visualize the digit images and model results.

---

## Scikit-learn
- A widely used library for classical machine learning in Python.
- We will use it for:
  - splitting the dataset,
  - building k-Nearest Neighbors classifiers,
  - evaluating model performance.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## From Images to Features

Machine learning algorithms work with numerical representations of data.

Let's start with a small image dataset and see what an image looks like from the perspective of a machine learning algorithm.

### Data Loading

In [ ]:
from sklearn.datasets import load_digits
digits = load_digits()

X = digits.data
y = digits.target
images = digits.images

print("Number of samples:", len(X))
print("X shape:", X.shape)  #every image is 8 x 8
print("y shape:", y.shape)
print("Image shape:", images[0].shape)

The dataset contains 8 × 8 grayscale images.

Therefore, each image contains:

$$
8 \times 8 = 64
$$

pixel values that can be used as features.

`load_digits()` conveniently provides these samples in two forms:

- `digits.images`: the original 8 × 8 image representation,
- `digits.data`: the same image already converted into a 64-dimensional feature vector.

Therefore, we do not need to manually flatten the images in this example.

As a convention in machine learning, a dataset is represented as matrix of size N x d, where N is the number of samples
in the dataset, and d is the dimensionality of the feature vector. All the models in scikit-learn follow this convention.

In [ ]:
sample_index = 0

plt.figure(figsize=(4, 4))
plt.imshow(images[sample_index], cmap="gray")
plt.title(f"Label: {y[sample_index]}")
plt.axis("off")
plt.show()

### How Is an Image Represented Numerically?

A grayscale image can be represented as a **matrix of pixel intensities**.

For an 8 × 8 image:

- each row of the matrix corresponds to one horizontal row of pixels,
- each column corresponds to a pixel position within that row,
- each number represents the intensity of that pixel.

In this dataset, pixel values range from **0 to 16**. Smaller values represent darker pixels, while larger values represent brighter pixels.

The same 8 × 8 matrix can also be written as a vector of 64 values so that it can be used as input to a machine learning algorithm.

In [ ]:
print("2D image representation:")
print(images[sample_index])

print("\nFlattened feature vector:")
print(X[sample_index])

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(10, 4))

for i, ax in enumerate(axes.ravel()):
    ax.imshow(images[i], cmap="gray")
    ax.set_title(f"Label: {y[i]}")
    ax.axis("off")

plt.tight_layout()
plt.show()

## Training, Validation, and Test Sets

Our goal is not simply to perform well on the data that the model has already seen.

We want the model to **generalize** to new, unseen samples.

For this reason, we divide the dataset into three parts:

- **Training set:** used to fit the model
- **Validation set:** used to compare model choices and hyperparameters
- **Test set:** used only at the end to estimate final performance on unseen data

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.30,
    stratify=y,
    random_state=1231
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=1231
)

print("Training set:", X_train.shape, y_train.shape)
print("Validation set:", X_val.shape, y_val.shape)
print("Test set:", X_test.shape, y_test.shape)

We used: stratify=y when splitting the dataset.
This helps preserve approximately the same class proportions in each split.
For example, if around 10% of the original samples belong to class 3, approximately the same proportion should appear in the training, validation, and test sets.

## How Does k-Nearest Neighbors Work?

Now that we have training, validation, and test sets, let's see how a k-Nearest Neighbors classifier makes a prediction.

The basic idea is simple:

1. Take a new sample.
2. Measure its distance to the training samples.
3. Find the **k nearest neighbors**.
4. Predict the class using the labels of those neighbors.

We will first perform these steps manually for a single image.

In [ ]:
query_idx = 168

query = X_val[query_idx]
true_label = y_val[query_idx]

plt.figure(figsize=(4, 4))
plt.imshow(query.reshape(8, 8), cmap="gray", vmin=0, vmax=16)
plt.title("Query Image")
plt.axis("off")
plt.show()

Imagine that the above image is a new sample that our model has not seen during training.

Before using kNN, what digit do you think it represents?

### Measuring Similarity with Distance

kNN assumes that samples that are **close to each other in the feature space** are likely to have similar labels.

A common distance measure is the **Euclidean distance**.

For two samples represented by feature vectors $x$ and $q$:

$$
d(x,q)
=
\sqrt{
\sum_{j=1}^{d}
(x_j-q_j)^2
}
$$

In our dataset, each image has 64 features, so the distance is calculated using all 64 pixel values.

A **smaller distance** means that two images are considered more similar by kNN.

In [ ]:
distances = np.linalg.norm(X_train - query, axis=1)

print("Number of distances calculated:", len(distances))
print("Smallest distance:", distances.min())
print("Largest distance:", distances.max())

### Finding the Nearest Neighbors

Once the distances have been calculated, we can sort the training samples from the closest to the farthest.

Let's inspect the five nearest neighbors of our query image.

In [ ]:
nearest_indices = np.argsort(distances)[:5]

nearest_labels = y_train[nearest_indices]
nearest_distances = distances[nearest_indices]

print("Nearest labels:", nearest_labels)
print("Distances:", np.round(nearest_distances, 2))

In [ ]:
fig, axes = plt.subplots(1, 6, figsize=(13, 2.5))

# Query image
axes[0].imshow(query.reshape(8, 8), cmap="gray", vmin=0, vmax=16)
axes[0].set_title("Query")
axes[0].axis("off")

# Five nearest neighbors
for i, idx in enumerate(nearest_indices):
    axes[i + 1].imshow(
        X_train[idx].reshape(8, 8),
        cmap="gray",
        vmin=0,
        vmax=16
    )
    axes[i + 1].set_title(
        f"Neighbor {i+1}\nLabel: {y_train[idx]}"
    )
    axes[i + 1].axis("off")

plt.tight_layout()
plt.show()

### Majority Voting

For $k>1$, kNN considers multiple neighbors.

In classification, the most common approach is **majority voting**: the class that appears most frequently among the $k$ nearest neighbors becomes the prediction.

In [ ]:
for k in [1, 3, 5]:
    labels = nearest_labels[:k]
    prediction = np.bincount(labels).argmax()

    print(f"k = {k}")
    print("Neighbor labels:", labels)
    print("Prediction:", prediction)
    print()

## kNN with Scikit-Learn

We have just implemented the main idea of kNN manually:

**distance → nearest neighbors → majority vote → prediction**

Scikit-learn provides the `KNeighborsClassifier` class to perform the same procedure for us.

Let's start with $k=5$.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

knn = KNeighborsClassifier(
    n_neighbors=5,
    metric="euclidean"
)

knn.fit(X_train, y_train)

For many machine learning models, `fit()` means estimating parameters from the training data.

kNN is different.

It is called a **lazy learning algorithm** because it does not learn a set of model parameters during training. Instead, it mainly stores the training samples and performs most of its computation when a prediction is requested.

In [ ]:
prediction = knn.predict(query.reshape(1, -1))

print("True label:", true_label)
print("Predicted label:", prediction[0])

In [ ]:
print("Before reshape:", query.shape)
print("After reshape:", query.reshape(1, -1).shape)

### Evaluating the Model

A single correct prediction does not tell us whether the model is good.

We should evaluate the model on many unseen samples.

For classification, one simple performance measure is **accuracy**:

$$
\text{Accuracy}
=
\frac{\text{Number of correct predictions}}
{\text{Total number of predictions}}
$$

In [ ]:
from sklearn.metrics import accuracy_score
y_val_pred = knn.predict(X_val)

val_accuracy = accuracy_score(y_val, y_val_pred)

print(f"Validation Accuracy: {val_accuracy:.3f}")

## Choosing Hyperparameters for kNN

The value of $k$ is one of the most important **hyperparameters** of kNN.

- A very small $k$ makes predictions depend strongly on individual training samples.
- A larger $k$ uses information from more neighbors and usually produces smoother predictions.
- If $k$ becomes too large, local structure may be lost.

However, $k$ is not the only choice we make.

The **distance metric** determines how the distance between two samples is calculated and therefore affects which samples are considered "nearest".

We will compare two commonly used distance metrics:

- **Euclidean distance**
- **Manhattan distance**

We will use the **validation set** to compare different combinations of $k$ and distance metric.

For two samples $x$ and $q$:

**Euclidean distance**

$$
d(x,q)
=
\sqrt{
\sum_{j=1}^{d}
(x_j-q_j)^2
}
$$

**Manhattan distance**

$$
d(x,q)
=
\sum_{j=1}^{d}
|x_j-q_j|
$$

Both measure how far two samples are from each other, but they define distance differently.

Therefore, the same query sample may have different nearest neighbors depending on the chosen metric.

In [ ]:
k_values = [1, 3, 5, 7, 9, 15]
metrics = ["euclidean", "manhattan"]

results = []

for metric in metrics:
    for k in k_values:

        model = KNeighborsClassifier(
            n_neighbors=k,
            metric=metric
        )

        model.fit(X_train, y_train)

        train_accuracy = model.score(X_train, y_train)
        val_accuracy = model.score(X_val, y_val)

        results.append({
            "k": k,
            "metric": metric,
            "train_accuracy": train_accuracy,
            "val_accuracy": val_accuracy
        })

        print(
            f"metric={metric:9s} | "
            f"k={k:2d} | "
            f"Train Accuracy: {train_accuracy:.3f} | "
            f"Validation Accuracy: {val_accuracy:.3f}"
        )

In [ ]:
plt.figure(figsize=(7, 4))

for metric in metrics:

    metric_results = [
        result for result in results
        if result["metric"] == metric
    ]

    val_accuracies = [
        result["val_accuracy"]
        for result in metric_results
    ]

    plt.plot(
        k_values,
        val_accuracies,
        marker="o",
        label=metric.capitalize()
    )

plt.xlabel("k")
plt.ylabel("Validation Accuracy")
plt.title("Effect of k and Distance Metric")
plt.xticks(k_values)
plt.legend()
plt.grid(alpha=0.3)

plt.show()

In [ ]:
best_result = max(
    results,
    key=lambda result: result["val_accuracy"]
)

best_k = best_result["k"]
best_metric = best_result["metric"]

print("Best k:", best_k)
print("Best metric:", best_metric)
print(
    f"Best Validation Accuracy: "
    f"{best_result['val_accuracy']:.3f}"
)

## Final Model

We have now finished hyperparameter selection.

The validation set has served its purpose, so we can combine the training and validation samples and use them to fit the final model.

The test set will remain untouched until the final evaluation.

In [ ]:
X_train_final = np.concatenate([X_train, X_val], axis=0)

y_train_final = np.concatenate([y_train, y_val], axis=0)

print("Original training set:", X_train.shape)
print("Final training set:", X_train_final.shape)

In [ ]:
final_knn = KNeighborsClassifier(
    n_neighbors=best_k,
    metric=best_metric
)

final_knn.fit(
    X_train_final,
    y_train_final
)

### Final Evaluation

The final model can now be evaluated on the test set.

Because the test samples were not used for either training or hyperparameter selection, this gives us an estimate of how the model performs on unseen data.

In [ ]:
from sklearn.metrics import accuracy_score

y_test_pred = final_knn.predict(X_test)

test_accuracy = accuracy_score(
    y_test,
    y_test_pred
)

print(f"Test Accuracy: {test_accuracy:.3f}")